In [1]:
# ============================================================
# Cell 1 — Install / verify dependencies
# ============================================================
!pip install torch torchvision torchaudio --quiet
!pip install numpy pandas scikit-learn matplotlib --quiet
print('All dependencies ready.')

All dependencies ready.


In [2]:
# ============================================================
# Cell 3 — Imports
# ============================================================
import os, math, random, warnings, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
PyTorch: 2.10.0+cu128


In [3]:
# ============================================================
# Cell 4 — Configuration  (edit DATA_PATH if needed)
# ============================================================

DATA_PATH    = '/content/ceemdan_components.csv'
N_EXPERIMENTS = 50
OUTPUT_CSV   = '/content/experiment_results.csv'
OUTPUT_TXT   = '/content/experiment_summary.txt'

# Component groupings
HF_IMFS      = ['IMF1', 'IMF2', 'IMF3', 'IMF6', 'IMF7']
LF_IMFS      = ['IMF4', 'IMF5', 'IMF8']
RESIDUAL_COL = ['Residual']
TARGET_COL   = 'Close'
ALL_COMPONENTS = HF_IMFS + LF_IMFS + RESIDUAL_COL

# Informer hyperparameters  (fixed — do not modify)
INF_SEQ_LEN    = 96
INF_LABEL_LEN  = 48
INF_PRED_LEN   = 1
INF_D_MODEL    = 256
INF_ENC_LAYERS = 2
INF_DEC_LAYERS = 1
INF_BATCH      = 32
INF_LR         = 1e-4
INF_EPOCHS     = 50
N_HEADS        = 8
D_FF           = INF_D_MODEL * 4
DROPOUT        = 0.05

# LSTM hyperparameters  (fixed — do not modify)
LSTM_LOOK_BACK = 20
LSTM_DEPTH     = 4
LSTM_HIDDEN    = 4
LSTM_DROPOUT   = 0.1
LSTM_EPOCHS    = 100
LSTM_BATCH     = 32
LSTM_LR        = 1e-3

RESIDUAL_STD_THRESHOLD = 1e-6

print('Config loaded.')

Config loaded.


In [4]:
# ============================================================
# Cell 5 — Utilities: seed, windows, datasets
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def create_lstm_windows(series, look_back):
    X, y = [], []
    for i in range(len(series) - look_back):
        X.append(series[i: i + look_back])
        y.append(series[i + look_back])
    return (np.array(X, dtype=np.float32)[..., np.newaxis],
            np.array(y, dtype=np.float32))

def create_informer_windows(series, seq_len, label_len, pred_len):
    enc_x, dec_x, targets = [], [], []
    for i in range(len(series) - seq_len - pred_len + 1):
        enc_seq    = series[i: i + seq_len]
        label_part = series[i + seq_len - label_len: i + seq_len]
        dec_seq    = np.concatenate([label_part, np.zeros(pred_len, dtype=np.float32)])
        enc_x.append(enc_seq)
        dec_x.append(dec_seq)
        targets.append(series[i + seq_len])
    return (np.array(enc_x, dtype=np.float32)[..., np.newaxis],
            np.array(dec_x, dtype=np.float32)[..., np.newaxis],
            np.array(targets, dtype=np.float32))

class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.from_numpy(X), torch.from_numpy(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class InformerDataset(Dataset):
    def __init__(self, enc_x, dec_x, targets):
        self.enc_x   = torch.from_numpy(enc_x)
        self.dec_x   = torch.from_numpy(dec_x)
        self.targets = torch.from_numpy(targets)
    def __len__(self): return len(self.targets)
    def __getitem__(self, i): return self.enc_x[i], self.dec_x[i], self.targets[i]

def compute_metrics(y_true, y_pred, label=''):
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    rmse   = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae    = float(mean_absolute_error(y_true, y_pred))
    mask   = np.abs(y_true) > 1e-8
    mape   = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print('Utilities defined.')

Utilities defined.


In [5]:
# ============================================================
# Cell 6 — Model definitions (Informer + LSTM)
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) *
                        (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class ProbSparseAttention(nn.Module):
    def __init__(self, d_model, n_heads, factor=5, attention_dropout=0.05):
        super().__init__()
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.dropout  = nn.Dropout(attention_dropout)
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, queries, keys, values, attn_mask=None):
        B, L_Q, _ = queries.shape
        L_K       = keys.shape[1]
        Q = self.q_proj(queries).view(B, L_Q, self.n_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(keys).view(B, L_K, self.n_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(values).view(B, L_K, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, -1e9)
        out = torch.matmul(self.dropout(torch.softmax(scores, dim=-1)), V)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, L_Q, -1))

class InformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()
        self.attn  = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                    nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm1(x + self.drop(self.attn(x, x, x)))
        return self.norm2(x + self.drop(self.ff(x)))

class InformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()
        self.self_attn  = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.cross_attn = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.ff         = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x, enc_out):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x)))
        x = self.norm2(x + self.drop(self.cross_attn(x, enc_out, enc_out)))
        return self.norm3(x + self.drop(self.ff(x)))

class Informer(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Linear(1, INF_D_MODEL)
        self.dec_embed = nn.Linear(1, INF_D_MODEL)
        self.enc_pos   = PositionalEncoding(INF_D_MODEL, INF_SEQ_LEN + 10, DROPOUT)
        self.dec_pos   = PositionalEncoding(INF_D_MODEL, INF_LABEL_LEN + INF_PRED_LEN + 10, DROPOUT)
        self.encoder   = nn.ModuleList([InformerEncoderLayer(INF_D_MODEL, N_HEADS, D_FF, DROPOUT)
                                        for _ in range(INF_ENC_LAYERS)])
        self.decoder   = nn.ModuleList([InformerDecoderLayer(INF_D_MODEL, N_HEADS, D_FF, DROPOUT)
                                        for _ in range(INF_DEC_LAYERS)])
        self.enc_norm  = nn.LayerNorm(INF_D_MODEL)
        self.dec_norm  = nn.LayerNorm(INF_D_MODEL)
        self.proj      = nn.Linear(INF_D_MODEL, 1)
    def forward(self, enc_x, dec_x):
        enc_out = self.enc_pos(self.enc_embed(enc_x))
        for layer in self.encoder: enc_out = layer(enc_out)
        enc_out = self.enc_norm(enc_out)
        dec_out = self.dec_pos(self.dec_embed(dec_x))
        for layer in self.decoder: dec_out = layer(dec_out, enc_out)
        return self.proj(self.dec_norm(dec_out)[:, -INF_PRED_LEN:, :]).squeeze(-1).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, LSTM_HIDDEN, LSTM_DEPTH, batch_first=True,
                            dropout=LSTM_DROPOUT if LSTM_DEPTH > 1 else 0.0)
        self.fc   = nn.Linear(LSTM_HIDDEN, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

print('Model classes defined.')

Model classes defined.


In [6]:
# ============================================================
# Cell 7 — Training functions
# ============================================================

def _train_loop(model, train_dl, val_dl, optimizer, criterion, epochs, patience):
    best_val, best_w, no_imp = float('inf'), None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_dl:
            optimizer.zero_grad()
            if len(batch) == 3:
                enc_b, dec_b, y_b = (t.to(DEVICE) for t in batch)
                loss = criterion(model(enc_b, dec_b), y_b)
            else:
                X_b, y_b = (t.to(DEVICE) for t in batch)
                loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        vl_losses = []
        with torch.no_grad():
            for batch in val_dl:
                if len(batch) == 3:
                    enc_b, dec_b, y_b = (t.to(DEVICE) for t in batch)
                    vl_losses.append(criterion(model(enc_b, dec_b), y_b).item())
                else:
                    X_b, y_b = (t.to(DEVICE) for t in batch)
                    vl_losses.append(criterion(model(X_b), y_b).item())
        vl = float(np.mean(vl_losses))
        if vl < best_val:
            best_val = vl
            best_w   = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            no_imp   = 0
        else:
            no_imp += 1
            if no_imp >= patience: break
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_w.items()})
    return model

def train_informer(comp, scaled_train, scaled_val, scaled_test, patience=7):
    tr, vl, te = scaled_train[comp], scaled_val[comp], scaled_test[comp]
    enc_tr, dec_tr, y_tr = create_informer_windows(tr, INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)
    enc_vl, dec_vl, y_vl = create_informer_windows(
        np.concatenate([tr[-INF_SEQ_LEN:], vl]), INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)
    enc_te, dec_te, y_te = create_informer_windows(
        np.concatenate([vl[-INF_SEQ_LEN:], te]), INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)
    if len(enc_tr) == 0: return None, None
    pin = DEVICE.type == 'cuda'
    train_dl = DataLoader(InformerDataset(enc_tr, dec_tr, y_tr), INF_BATCH, shuffle=False, pin_memory=pin)
    val_dl   = DataLoader(InformerDataset(enc_vl, dec_vl, y_vl), INF_BATCH, shuffle=False)
    test_dl  = DataLoader(InformerDataset(enc_te, dec_te, y_te), INF_BATCH, shuffle=False)
    model = _train_loop(Informer().to(DEVICE), train_dl, val_dl,
                        Adam(Informer().to(DEVICE).parameters(), lr=INF_LR),
                        nn.MSELoss(), INF_EPOCHS, patience)
    # re-create optimizer bound to the actual trained model
    model = _train_loop(Informer().to(DEVICE), train_dl, val_dl,
                        Adam(model.parameters() if hasattr(model,'parameters') else Informer().to(DEVICE).parameters(), lr=INF_LR),
                        nn.MSELoss(), INF_EPOCHS, patience)
    model.eval()
    preds = []
    with torch.no_grad():
        for enc_b, dec_b, _ in test_dl:
            preds.append(model(enc_b.to(DEVICE), dec_b.to(DEVICE)).cpu().numpy())
    return model, np.concatenate(preds)

def train_lstm(comp, scaled_train, scaled_val, scaled_test, patience=10):
    tr, vl, te = scaled_train[comp], scaled_val[comp], scaled_test[comp]
    X_tr, y_tr = create_lstm_windows(tr, LSTM_LOOK_BACK)
    X_vl, y_vl = create_lstm_windows(np.concatenate([tr[-LSTM_LOOK_BACK:], vl]), LSTM_LOOK_BACK)
    X_te, y_te = create_lstm_windows(np.concatenate([vl[-LSTM_LOOK_BACK:], te]), LSTM_LOOK_BACK)
    pin = DEVICE.type == 'cuda'
    train_dl = DataLoader(LSTMDataset(X_tr, y_tr), LSTM_BATCH, shuffle=False, pin_memory=pin)
    val_dl   = DataLoader(LSTMDataset(X_vl, y_vl), LSTM_BATCH, shuffle=False)
    test_dl  = DataLoader(LSTMDataset(X_te, y_te), LSTM_BATCH, shuffle=False)
    model = LSTMModel().to(DEVICE)
    model = _train_loop(model, train_dl, val_dl,
                        Adam(model.parameters(), lr=LSTM_LR),
                        nn.MSELoss(), LSTM_EPOCHS, patience)
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, _ in test_dl:
            preds.append(model(X_b.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)

def train_imf8_lstm(imf8_train_raw, imf8_val_raw, imf8_test_raw, patience=10):
    """First-difference cumsum LSTM pipeline for IMF8."""
    delta_train = np.diff(imf8_train_raw)
    delta_val   = np.diff(np.concatenate([[imf8_train_raw[-1]], imf8_val_raw]))
    delta_test  = np.diff(np.concatenate([[imf8_val_raw[-1]],   imf8_test_raw]))
    scaler = StandardScaler()
    scaler.fit(delta_train.reshape(-1, 1))
    dtr = scaler.transform(delta_train.reshape(-1, 1)).flatten().astype(np.float32)
    dvl = scaler.transform(delta_val.reshape(-1,   1)).flatten().astype(np.float32)
    dte = scaler.transform(delta_test.reshape(-1,  1)).flatten().astype(np.float32)
    X_tr, y_tr = create_lstm_windows(dtr, LSTM_LOOK_BACK)
    X_vl, y_vl = create_lstm_windows(np.concatenate([dtr[-LSTM_LOOK_BACK:], dvl]), LSTM_LOOK_BACK)
    X_te, y_te = create_lstm_windows(np.concatenate([dvl[-LSTM_LOOK_BACK:], dte]), LSTM_LOOK_BACK)
    pin = DEVICE.type == 'cuda'
    train_dl = DataLoader(LSTMDataset(X_tr, y_tr), LSTM_BATCH, shuffle=False, pin_memory=pin)
    val_dl   = DataLoader(LSTMDataset(X_vl, y_vl), LSTM_BATCH, shuffle=False)
    test_dl  = DataLoader(LSTMDataset(X_te, y_te), LSTM_BATCH, shuffle=False)
    model = LSTMModel().to(DEVICE)
    model = _train_loop(model, train_dl, val_dl,
                        Adam(model.parameters(), lr=LSTM_LR),
                        nn.MSELoss(), LSTM_EPOCHS, patience)
    model.eval()
    delta_preds = []
    with torch.no_grad():
        for X_b, _ in test_dl:
            delta_preds.append(model(X_b.to(DEVICE)).cpu().numpy())
    delta_preds_orig = scaler.inverse_transform(
        np.concatenate(delta_preds).reshape(-1, 1)).flatten().astype(np.float64)
    return float(imf8_val_raw[-1]) + np.cumsum(delta_preds_orig)

print('Training functions defined.')

Training functions defined.


In [7]:
# ============================================================
# Cell 8 — Single-experiment runner
# ============================================================

def run_experiment(df_train, df_val, df_test, seed):
    set_seed(seed)

    # Scaling
    scalers, scaled_train, scaled_val, scaled_test = {}, {}, {}, {}
    for col in ALL_COMPONENTS:
        if col == 'IMF8': continue
        sc = MinMaxScaler(feature_range=(0, 1))
        sc.fit(df_train[[col]].values)
        scaled_train[col] = sc.transform(df_train[[col]].values).flatten()
        scaled_val[col]   = sc.transform(df_val[[col]].values).flatten()
        scaled_test[col]  = sc.transform(df_test[[col]].values).flatten()
        scalers[col] = sc

    def safe_inv(name, preds_sc):
        return scalers[name].inverse_transform(
            preds_sc.reshape(-1,1)).flatten().astype(np.float64)

    # HF → Informer
    inf_preds_orig = {}
    for imf in HF_IMFS:
        _, preds = train_informer(imf, scaled_train, scaled_val, scaled_test)
        if preds is not None:
            inf_preds_orig[imf] = safe_inv(imf, preds)

    # LF → LSTM
    lstm_preds_orig = {}
    INCLUDE_RESIDUAL = df_train['Residual'].std() > RESIDUAL_STD_THRESHOLD
    for comp in LF_IMFS + RESIDUAL_COL:
        if comp == 'IMF8': continue
        if comp == 'Residual' and not INCLUDE_RESIDUAL: continue
        lstm_preds_orig[comp] = safe_inv(comp, train_lstm(comp, scaled_train, scaled_val, scaled_test))

    # IMF8 first-difference pipeline
    lstm_preds_orig['IMF8'] = train_imf8_lstm(
        df_train['IMF8'].values.astype(np.float64),
        df_val['IMF8'].values.astype(np.float64),
        df_test['IMF8'].values.astype(np.float64))

    # Hybrid aggregation
    test_len = len(df_test)
    y_hat  = sum(inf_preds_orig.get(imf, np.zeros(test_len)) for imf in HF_IMFS)
    y_hat += sum(lstm_preds_orig.get(c, np.zeros(test_len)) for c in LF_IMFS + RESIDUAL_COL)
    y_true = df_test[TARGET_COL].values.astype(np.float64)

    return compute_metrics(y_true, y_hat, label='Hybrid')

print('run_experiment() defined.')

run_experiment() defined.


In [8]:
# ============================================================
# Cell 9 — Load data & chronological split
# ============================================================

df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

train_mask    = df['Date'].dt.year <= 2022
test_mask     = df['Date'].dt.year >= 2023
df_train_full = df[train_mask].reset_index(drop=True)
df_test       = df[test_mask].reset_index(drop=True)
val_size      = int(len(df_train_full) * 0.10)
df_train      = df_train_full.iloc[:-val_size].reset_index(drop=True)
df_val        = df_train_full.iloc[-val_size:].reset_index(drop=True)

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df.Date.min().date()} → {df.Date.max().date()}')
print(f'Train : {len(df_train)}  Val : {len(df_val)}  Test : {len(df_test)}')

Dataset shape : (2599, 11)
Date range    : 2015-01-09 → 2025-07-25
Train : 1770  Val : 196  Test : 633


In [9]:
# ============================================================
# Cell 10 — Run all 50 experiments  (⚠ may take a while on GPU)
# ============================================================

# Reproducible seed list (always the same 50 seeds regardless of run order)
rng   = random.Random(0)
seeds = [rng.randint(0, 99999) for _ in range(N_EXPERIMENTS)]

all_results = []

for exp_idx, seed in enumerate(seeds, start=1):
    print(f'\n[{exp_idx:02d}/{N_EXPERIMENTS}]  seed={seed}', flush=True)
    m = run_experiment(df_train, df_val, df_test, seed)
    m['experiment'] = exp_idx
    m['seed']       = seed
    all_results.append(m)
    print(f'  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  MAPE={m["MAPE"]:.4f}%', flush=True)

print('\n✅ All experiments complete.')


[01/50]  seed=50494
  RMSE=5010.3959  MAE=4924.5944  MAPE=22.4395%

[02/50]  seed=99346
  RMSE=1333.6635  MAE=1186.5204  MAPE=5.7140%

[03/50]  seed=55125
  RMSE=4567.5670  MAE=4375.3210  MAPE=19.6394%

[04/50]  seed=5306
  RMSE=5390.4570  MAE=5204.1363  MAPE=23.4331%

[05/50]  seed=33936
  RMSE=3838.4186  MAE=3636.6792  MAPE=16.2406%

[06/50]  seed=67013
  RMSE=3274.0803  MAE=2939.8991  MAPE=12.9026%

[07/50]  seed=63691
  RMSE=2134.1359  MAE=1768.1481  MAPE=7.5781%

[08/50]  seed=53075
  RMSE=3244.9379  MAE=2896.8918  MAPE=12.6884%

[09/50]  seed=39755
  RMSE=2958.2893  MAE=2532.4025  MAPE=12.6410%

[10/50]  seed=62468
  RMSE=3040.0525  MAE=2640.4134  MAPE=11.4712%

[11/50]  seed=46930
  RMSE=1989.7608  MAE=1524.3478  MAPE=7.7767%

[12/50]  seed=76465
  RMSE=4830.8807  MAE=4523.1870  MAPE=20.1116%

[13/50]  seed=28631
  RMSE=3728.7555  MAE=3411.6126  MAPE=15.0520%

[14/50]  seed=66150
  RMSE=5245.1782  MAE=5041.2298  MAPE=22.6717%

[15/50]  seed=18254
  RMSE=3576.4452  MAE=3229.8756

In [10]:
# ============================================================
# Cell 11 — Save results & print summary
# ============================================================

import csv, os

# --- CSV ---
with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['experiment','seed','RMSE','MAE','MAPE','label'])
    writer.writeheader()
    writer.writerows(all_results)
print(f'Per-experiment results  → {OUTPUT_CSV}')

# --- Summary ---
rmses = [r['RMSE'] for r in all_results]
maes  = [r['MAE']  for r in all_results]
mapes = [r['MAPE'] for r in all_results]

lines = [
    '=' * 55,
    '  MODULE 3 — 50-EXPERIMENT SUMMARY',
    '=' * 55,
    f'  Experiments : {N_EXPERIMENTS}',
    f'  {"Metric":<10} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}',
    '-' * 55,
    f'  {"RMSE":<10} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}',
    f'  {"MAE":<10} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}',
    f'  {"MAPE(%)":<10} {np.mean(mapes):>10.4f} {np.std(mapes):>10.4f} {np.min(mapes):>10.4f} {np.max(mapes):>10.4f}',
    '=' * 55,
]
summary_text = '\n'.join(lines)
print('\n' + summary_text)

with open(OUTPUT_TXT, 'w') as f:
    f.write(summary_text + '\n')
print(f'\nSummary                 → {OUTPUT_TXT}')

Per-experiment results  → /content/experiment_results.csv

  MODULE 3 — 50-EXPERIMENT SUMMARY
  Experiments : 50
  Metric           Mean        Std        Min        Max
-------------------------------------------------------
  RMSE        3523.1563  1332.5851  1063.0754  7483.5636
  MAE         3237.8674  1373.5250   920.7965  7331.5383
  MAPE(%)       14.4908     6.1587     4.4756    33.3051

Summary                 → /content/experiment_summary.txt
